# 07 · Rolling chunks up to documents

Most long documents are split into chunks, labelled chunk by chunk, and then summarised somehow. Does the rule matter, and
is reading every chunk better than reading the opening?

- **first chunk only:** the opening alone, as the shorter methods do.
- **any chunk:** a label is kept if any chunk has it.
- **share of chunks:** a label is kept if at least 20% of chunks have it.
- **certainty weighted:** log-odds summed across chunks, the rule TEF found best (arXiv 2609.27165).

Chunks are labelled with the main model, through the funnel with label cards. The small model was dropped here after the
pilot, because it ran out of tokens on most chunks. Chunks of 400 words, with at most 60 spread
evenly through a document.

**Limits.** These are the longest EU laws: all those over 20,000 words, topped up from those over 5,000. Laws are not
evaluation reports, and long laws are often long because of annexes and tables. HumSet's "documents" are only the excerpts
analysts selected, with gold labels that are the union of theirs, so it shows how rules behave on already-relevant text.

**What would count as what, written before running.**
- Expected from TEF: certainty weighting beats simple rules. TEF tested this on short posts, so this is an open question
  on long documents.
- Expected: any-chunk has high recall and low precision.
- Differences count only where the 95% paired bootstrap interval excludes zero.


In [1]:
import pandas as pd

from tiered_taxonomy import experiment as ex
from tiered_taxonomy import runs
from tiered_taxonomy.config import data_root, load_config

root = data_root()
config = load_config()
seed = config["seed"]
samples = config["samples"]


def sample_for(data):
    if data.name == ex.EURLEX:
        settings = samples["eurlex"]
        return ex.sample_eurlex(
            data,
            settings["normal_documents"],
            settings["long_documents"],
            settings["long_threshold_words"],
            seed,
        )
    return ex.sample_humset(data, samples["humset"]["per_language"], seed)


def table(results):
    rows = []
    for name, result in results.items():
        scores = result["scores"]
        row = {
            "method": name,
            "labels per item": scores["labels_per_item"],
            "exact F1": round(scores["exact"]["f1"], 3),
            "exact precision": round(scores["exact"]["precision"], 3),
            "exact recall": round(scores["exact"]["recall"], 3),
        }
        row.update({f"tier {tier} F1": round(value["f1"], 3) for tier, value in scores["tiers"].items()})
        row.update({f"{band} recall": value["recall"] for band, value in scores["rarity"].items()})
        if "run" in result:
            run = result["run"]
            row.update(
                {
                    "calls per item": round(run["calls"] / run["items"], 1),
                    "cost per 1000 items ($)": round(1000 * run["cost_usd"] / run["items"], 2),
                    "malformed": run["malformed"],
                    "invalid": run["invalid"],
                }
            )
        rows.append(row)
    return pd.DataFrame(rows)

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from tiered_taxonomy.datasets import chunk_words
from tiered_taxonomy.methods.rollup import evenly_spaced

eurlex = ex.load_eurlex(root)
humset = ex.load_humset(root)
settings = samples["eurlex"]
bench = runs.make_bench(eurlex, root, config)
runs.load_cards(bench)
laws = runs.rollup_sample_eurlex(eurlex, settings["rollup_documents"], settings["long_threshold_words"], seed)
documents = [
    (law, evenly_spaced(chunk_words(law.text, config["text"]["chunk_words"]), settings["rollup_max_chunks"]))
    for law in laws
]
results = {ex.EURLEX: runs.rollup_compare(bench, documents)}

In [3]:
import numpy as np

bench_h = runs.make_bench(humset, root, config)
runs.load_cards(bench_h)
groups = runs.group_items(humset.test, samples["humset"]["rollup_min_entries"])
rng = np.random.default_rng(seed)
chosen = sorted(
    rng.choice(sorted(groups), size=samples["humset"]["rollup_documents"], replace=False).tolist()
)
documents_h = [
    (runs.merge_group(key, groups[key], humset.taxonomy), [entry.text for entry in groups[key]])
    for key in chosen
]
results[ex.HUMSET] = runs.rollup_compare(bench_h, documents_h)

In [4]:
def rollup_table(result):
    rows = []
    for rule in ("first_chunk_only", "any_chunk", "share_of_chunks", "certainty_weighted"):
        scores = result[rule]
        row = {
            "rule": rule,
            "labels per doc": scores["labels_per_item"],
            "exact F1": round(scores["exact"]["f1"], 3),
            "precision": round(scores["exact"]["precision"], 3),
            "recall": round(scores["exact"]["recall"], 3),
        }
        row.update({f"tier {tier} F1": round(value["f1"], 3) for tier, value in scores["tiers"].items()})
        rows.append(row)
    return pd.DataFrame(rows)


print({key: results[ex.EURLEX][key] for key in ("documents", "chunks", "cost_usd", "malformed_chunks")})
rollup_table(results[ex.EURLEX])

{'documents': 120, 'chunks': 5289, 'cost_usd': 3.1187, 'malformed_chunks': 51}


,rule,labels per doc,exact F1,precision,recall,tier 1 F1,tier 2 F1,tier 3 F1
0,first_chunk_only,3.83,0.156,0.216,0.122,0.413,0.291,0.229
1,any_chunk,55.04,0.113,0.063,0.517,0.548,0.362,0.256
2,share_of_chunks,3.17,0.219,0.342,0.161,0.474,0.352,0.279
3,certainty_weighted,55.02,0.113,0.063,0.517,0.548,0.362,0.256


In [5]:
print({key: results[ex.HUMSET][key] for key in ("documents", "chunks", "cost_usd", "malformed_chunks")})
rollup_table(results[ex.HUMSET])

{'documents': 200, 'chunks': 4340, 'cost_usd': 1.3112, 'malformed_chunks': 0}


,rule,labels per doc,exact F1,precision,recall,tier 1 F1,tier 2 F1
0,first_chunk_only,1.66,0.161,0.605,0.093,0.219,0.137
1,any_chunk,11.74,0.545,0.523,0.569,0.675,0.451
2,share_of_chunks,2.98,0.266,0.613,0.169,0.350,0.210
3,certainty_weighted,11.74,0.545,0.523,0.569,0.675,0.451


In [6]:
ex.save_results(
    "07_rollup",
    {"results": results, "models": config["models"], "chunk_words": config["text"]["chunk_words"]},
);